# Detection d'anomalies salariales par Machine Learning (approche non supervisee)

Objectif : a partir des **3 fichiers de donnees bruts** (`input/employes.csv`, `input/market.csv`,
`input/bands.csv`), reconstruire les memes indicateurs de comparaison marche que le moteur de regles
existant (`src/anomaly_core.py`) puis detecter les anomalies salariales avec des **modeles ML non
supervises** (IsolationForest, Local Outlier Factor, One-Class SVM) : on ne dispose ici d'aucune
etiquette "Severity" prete a l'emploi (elle n'existe que dans `anomalies.csv`, produit par le moteur
de regles), donc il ne s'agit pas de reproduire un score de regles mais de laisser des algorithmes de
detection d'outliers trouver eux-memes les profils atypiques.

Plan :
1. Chargement des 3 fichiers bruts
2. Feature engineering (reprise des fonctions deja testees de `anomaly_core.py` : CompaRatio,
   RangePenetration, MarketRatio, statistiques de cohorte, PeerZ) - **sans** reprendre la partie
   "regles"/scoring du fichier
3. Analyse exploratoire (EDA) des features obtenues
4. Detection d'anomalies non supervisee (3 algorithmes compares)
5. Consensus multi-modeles et visualisation des anomalies detectees
6. Classification interpretable : un modele supervise (Random Forest) est entraine sur les labels
   produits par la detection non supervisee, pour en tirer une **feature importance** claire
7. Sauvegarde des resultats et du modele


## 1. Chargement des donnees brutes

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
plt.rcParams["figure.dpi"] = 100
RANDOM_STATE = 42

# Les fonctions de feature engineering (deja ecrites et testees dans le projet) vivent dans
# src/anomaly_core.py. On les reutilise plutot que de reecrire des formules qui existent deja.
from pathlib import Path
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / "src" / "anomaly_core.py").is_file())
sys.path.insert(0, str(ROOT / "src"))
MODEL_DIR = ROOT / "models"
OUTPUT_DIR = ROOT / "output" / "classification"
MODEL_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
from anomaly_core import bucket_anciennete, compute_features, apply_rulebook, cohort_stats_and_peers_nv

In [ ]:
employees = pd.read_csv(ROOT / "input" / "employes.csv", sep=";", encoding="cp1252")
market = pd.read_csv(ROOT / "input" / "market.csv", sep=";", encoding="cp1252")
bands = pd.read_csv(ROOT / "input" / "bands.csv", sep=";", encoding="cp1252")

print("employes :", employees.shape)
print("market   :", market.shape)
print("bands    :", bands.shape)
employees.head()

In [ ]:
employees.info()

## 2. Feature engineering

On reutilise `compute_features` (jointure avec `bands`/`market` sur `Job_Family`+`Grade`, calcul de
`CompaRatio`, `RangePenetration`, `MarketRatio`) et `cohort_stats_and_peers_nv` (statistiques de
cohorte + `PeerZ`, le z-score robuste vis-a-vis des pairs) de `anomaly_core.py`. `cohort_stats_and_peers_nv`
suppose que `Rule_Flags`/`Rule_Score` existent deja (normalement remplis par `apply_rulebook`) : on
appelle donc aussi `apply_rulebook` par simple dependance technique, puis on **supprime immediatement**
tout ce qui releve du scoring de regles (`Rule_Flags`, `Rule_Score`, `Reason_Principale`, `Peer_Flag`).
On ne reprend en revanche ni `aggregate_risk`, ni le `ml_anomaly` (IsolationForest) deja present dans ce
fichier : c'est precisement cette derniere etape qu'on refait nous-memes ci-dessous, en comparant
plusieurs algorithmes.

In [ ]:
df = compute_features(employees.copy(), bands, market)
df = apply_rulebook(df, rule_params={})
df = cohort_stats_and_peers_nv(df, rule_params={})

# Colonnes de scoring/regles (necessaires uniquement en interne a cohort_stats_and_peers_nv) :
# on les supprime, on ne garde que les statistiques de cohorte et le PeerZ, qui sont de simples
# features de comparaison, pas un verdict.
df = df.drop(columns=["Rule_Flags", "Rule_Score", "Peer_Flag", "Reason_Principale"], errors="ignore")

print(df.shape)
df[["Matricule", "Job_Family", "Grade", "Fixe_Annuel_MAD", "Min", "Mid", "Max", "Market_Median",
    "CompaRatio", "RangePenetration", "MarketRatio", "Cohort_Size", "PeerZ"]].head(10)

**Lecture du tableau ci-dessus.** La jointure a fonctionné : chaque employé a bien recu son `Min`/`Mid`/`Max` (bande interne) et son `Market_Median`. On remarque tout de suite que `CompaRatio` et `MarketRatio` sont **identiques** ligne par ligne (0.974197 / 0.974197, 1.898446 / 1.898446, ...) : ce n'est pas une coïncidence, c'est parce que dans ce jeu de données `bands.csv.Mid` est **rigoureusement égal** à `market.csv.Median` pour chaque combinaison `Job_Family`/`Grade` (vérifié : 100% des 91 lignes). Autrement dit, la "fourchette interne" et le "marché externe" racontent ici exactement la même histoire. Implication pour la suite : `CompaRatio` et `MarketRatio` n'apportent **aucune information supplémentaire l'une par rapport à l'autre** dans ce dataset (ce sera confirmé par une corrélation de 1.00 dans la matrice de corrélation plus bas) — dans un vrai jeu de données RH, ces deux ratios divergent generalement.

In [ ]:
missing = df.isna().sum()
missing[missing > 0].sort_values(ascending=False)

**Interprétation.** Seule `Nom` est vide (10000/10000) — c'est normal, ce champ n'a jamais été renseigné dans `employes.csv` et n'est pas utilisé. Aucune valeur manquante sur `CompaRatio`/`RangePenetration`/`MarketRatio`/`PeerZ` : cela **confirme que la jointure avec `bands`/`market` a réussi pour tous les employés**, donc que chaque combinaison `Job_Family`+`Grade` présente chez les employés existe bien dans `bands.csv` et `market.csv`.

In [ ]:
# Employes sans correspondance de bande/marche (Job_Family/Grade absent de bands ou market) :
# leurs ratios sont NaN et fausseraient l'entrainement ML. On les met de cote et on travaille sur
# le reste, qui couvre la grande majorite des effectifs.
before = len(df)
df = df.dropna(subset=["CompaRatio", "RangePenetration", "MarketRatio", "PeerZ"]).copy()
print(f"{before - len(df)} ligne(s) ecartee(s) faute de correspondance bande/marche -> {len(df)} lignes conservees")

**Interprétation.** `0 ligne(s) écartée(s)` : les 91 combinaisons `Job_Family`×`Grade` de `bands.csv`/`market.csv` couvrent bien la totalité des 10 000 employés de `employes.csv`. Rien n'a été perdu — on travaille sur la population complète pour toute la suite du notebook.

## 3. Analyse exploratoire (EDA)

**Objectif de ce graphe.** Verifier a quoi ressemble la variable la plus brute, le salaire (`Fixe_Annuel_MAD`) : est-elle tres etalee/asymetrique (justifiant une echelle log) ? Et est-ce que le niveau de salaire varie beaucoup d'un `Job_Family` a l'autre (justifiant de comparer chaque employe a son propre metier plutot qu'a la population entiere) ?

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
sns.histplot(df["Fixe_Annuel_MAD"], bins=50, kde=True, ax=axes[0], color="#4C72B0")
axes[0].set_title("Distribution du salaire annuel fixe (MAD)")

top_families = df["Job_Family"].value_counts().index
sns.boxplot(data=df, x="Job_Family", y="Fixe_Annuel_MAD", order=top_families, ax=axes[1])
axes[1].set_yscale("log")
axes[1].set_title("Salaire par Job_Family")
axes[1].tick_params(axis="x", rotation=75)
plt.tight_layout()
plt.show()

**Interprétation.** Le salaire va de 20 à 1953 (MAD, en milliers) avec une médiane à 212 et une moyenne à 248 : la distribution est **étalée à droite** (quelques très hauts salaires tirent la moyenne au-dessus de la médiane), d'où l'échelle log sur les boxplots. Le boxplot par `Job_Family` montre des niveaux très différents d'un métier à l'autre (ex. `IT`/`AUDIT` vs `RETAIL BANKING`) — **implication directe** : comparer un salaire brut entre deux `Job_Family` n'a pas de sens, il faut le rapporter à un référentiel propre à son métier et son grade. C'est exactement le rôle de `CompaRatio`/`MarketRatio` (regardés juste après) : ils neutralisent cet effet Job_Family/Grade.

**Objectif du graphe suivant.** Regarder la distribution des 4 ratios de comparaison (`CompaRatio`, `RangePenetration`, `MarketRatio`, `PeerZ`) : sont-ils bien centrés autour de leur valeur "normale" (1.0 pour les ratios, 0 pour PeerZ) ? Y a-t-il des valeurs extrêmes qui sortent du lot (candidates naturelles à être détectées comme anomalies) ?

In [ ]:
ratio_features = ["CompaRatio", "RangePenetration", "MarketRatio", "PeerZ"]

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
for ax, col in zip(axes.ravel(), ratio_features):
    sns.histplot(df[col], bins=50, kde=True, ax=ax, color="#55A868")
    ax.set_title(f"Distribution de {col}")
    ax.axvline(df[col].median(), color="red", linestyle="--", linewidth=1, label="mediane")
    ax.legend()
plt.tight_layout()
plt.show()

**Interprétation.** `CompaRatio` et `MarketRatio` ont **exactement la même médiane (1.004) et le même écart-type (0.217)**, avec 27.0% des employés en dessous de 0.85 et 27.2% au-dessus de 1.15 — les deux histogrammes sont visuellement superposables, ce qui confirme le constat de la section précédente (`Mid` = `Market_Median` dans ce dataset). `RangePenetration` est centrée vers 0.50 (le milieu de bande) mais avec 84.9% des employés en dessous de 0.85 (ce chiffre n'est pas comparable à celui de CompaRatio, l'échelle de RangePenetration est différente : 0 = Min, 1 = Max de la bande) — beaucoup d'employés se situent donc dans la moitié basse de leur bande. `PeerZ` est bien centré sur 0 (médiane exacte) avec seulement 1.05% des employés à |Z|≥2 et 0.36% à |Z|≥3 : peu d'employés sont des outliers francs par rapport à leurs pairs de cohorte — la majorité des écarts observés viennent donc plutôt du positionnement vs bande/marché (CompaRatio/MarketRatio) que d'un écart entre collègues très proches (PeerZ).

**Objectif du graphe suivant.** Une matrice de corrélation pour reperer (a) les variables redondantes entre elles (qui n'apporteraient rien de plus au modele si on les garde toutes), et (b) les variables qui se comportent ensemble ou a l'inverse independamment du salaire absolu.

In [ ]:
numeric_for_corr = [
    "Fixe_Annuel_MAD", "CompaRatio", "RangePenetration", "MarketRatio", "PeerZ",
    "Cohort_Size", "Cohort_Median", "Cohort_Mean", "Cohort_STD", "Cohort_P25", "Cohort_P75",
    "Age", "Anciennete", "Competence_N1", "Positionnement_9BOX",
]
corr = df[numeric_for_corr].corr()

plt.figure(figsize=(11, 9))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0, square=True)
plt.title("Matrice de correlation des variables numeriques")
plt.tight_layout()
plt.show()

**Interprétation.** Quelques lectures clés de cette matrice :
- `CompaRatio`, `RangePenetration` et `MarketRatio` sont corrélés à **1.00 entre eux, et même à 0.93 avec `PeerZ`** : dans ce dataset, ces 4 indicateurs (pourtant calculés de façons différentes — vs bande interne, vs marché externe, vs pairs de la cohorte) racontent en réalité **presque la même histoire**, à savoir l'écart du salaire de l'employé à ce qui est "attendu" pour son poste. C'est cohérent avec le fait que `Mid` = `Market_Median` vu plus haut, et que les cohortes sont elles-mêmes définies par Grade/Job_Family. **Implication forte pour le reste du notebook** : un modèle qui s'appuie sur ces 4 colonnes n'a en pratique qu'un seul vrai signal individuel à disposition, pas quatre — un point à garder en tête en lisant la feature importance en section 6.
- `Cohort_Median`, `Cohort_Mean`, `Cohort_P25`, `Cohort_P75` sont corrélés à **~0.90** avec `Fixe_Annuel_MAD` : logique, ce sont des agrégats calculés à partir du salaire lui-même au sein de la cohorte de l'employé — ce n'est pas un signal externe indépendant, c'est en partie une reformulation du salaire.
- `Cohort_Size` est corrélé **négativement (-0.33)** au salaire : les cohortes les plus nombreuses (ex. grades bas/métiers très peuplés comme `RETAIL BANKING`) correspondent aux salaires les plus bas — pas une anomalie, une caractéristique structurelle de l'organisation (pyramide des grades).
- `Age`, `Anciennete`, `Competence_N1`, `Positionnement_9BOX` sont **quasi non corrélés (~0.00-0.02)** avec le salaire brut : dans ce dataset, le salaire dépend presque exclusivement du couple Job_Family/Grade, pas du profil individuel de la personne. **Implication pour la détection d'anomalies** : ces variables RH ne seront probablement pas de bons signaux pour repérer un salaire anormal — elles servent plutôt à décrire le profil de la personne flaguée, pas à la flaguer.

**Objectif du graphe suivant.** Vérifier si certains groupes (métier, grade, entité, sexe, tranche d'ancienneté, hot job) ont un `CompaRatio` médian systématiquement plus haut ou plus bas que 1.0 — un signe qu'un groupe entier serait structurellement mieux/moins bien payé que sa bande, ce qui orienterait une lecture RH (équité) au-delà des cas individuels.

In [ ]:
cat_cols = ["Job_Family", "Grade", "Entite_N1", "Sexe", "Anciennete_Bucket", "Hot_job"]

fig, axes = plt.subplots(3, 2, figsize=(14, 15))
for ax, col in zip(axes.ravel(), cat_cols):
    med_compa = df.groupby(col)["CompaRatio"].median().sort_values(ascending=False)
    sns.barplot(x=med_compa.index.astype(str), y=med_compa.values, hue=med_compa.index.astype(str),
                palette="viridis", legend=False, ax=ax)
    ax.axhline(1.0, color="red", linestyle="--", linewidth=1)
    ax.set_title(f"CompaRatio median par {col}")
    ax.set_ylabel("CompaRatio median")
    ax.tick_params(axis="x", rotation=45)
plt.tight_layout()
plt.show()

**Interprétation.** Les écarts entre groupes sont **faibles** (tous les médians restent entre 0.96 et 1.03, à comparer à la référence 1.0) :
- Par `Job_Family` : de `CONFORMITE` (0.958, le plus bas) à `JURIDIQUE` (1.029, le plus haut) — moins de 4% d'écart entre le métier le mieux et le moins bien positionné vs sa bande.
- Par `Sexe` : `F` = 1.008 vs `M` = 0.999 — les femmes sont ici très légèrement **au-dessus** de leur bande par rapport aux hommes, pas en-dessous. Il n'y a donc pas de signal d'écart de rémunération défavorable aux femmes dans ce jeu de données (à confirmer avec l'analyse dédiée `gender_gap_analysis` du projet, plus rigoureuse sur ce sujet).
- Par `Anciennete_Bucket` : les plus jeunes dans le poste (0-2 ans, 1.015) sont legerement au-dessus des plus anciens (13-20 ans, 0.987) — pas d'effet flagrant de sous-paiement des nouveaux arrivants.

**Implication** : il n'y a pas de biais structurel massif d'un groupe entier — les anomalies qu'on va détecter seront donc probablement des cas **individuels** dispersés dans les groupes, plutôt que des groupes entiers mal positionnés.

## 4. Detection d'anomalies non supervisee

On construit une matrice de features (ratios de comparaison marche + stats de cohorte + profil RH),
standardisee/encodee, puis on entraine trois familles de detecteurs d'outliers :
- **Isolation Forest** (deja utilise ailleurs dans le projet, robuste en grande dimension)
- **Local Outlier Factor** (densite locale, detecte les outliers "relatifs a leur voisinage")
- **One-Class SVM** (frontiere de decision globale)

Chacun est parametre avec `contamination=0.05` (hypothese : ~5% de la population est anormale),
hypothese explicite a challenger metier, pas une verite mesuree.

In [ ]:
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor
from sklearn.svm import OneClassSVM
from sklearn.decomposition import PCA

numeric_features = [
    "CompaRatio", "RangePenetration", "MarketRatio", "PeerZ",
    "Cohort_Size", "Cohort_Median", "Cohort_Mean", "Cohort_STD", "Cohort_P25", "Cohort_P75",
    "Age", "Anciennete", "Competence_N1", "Positionnement_9BOX",
]
categorical_features = ["Grade", "Job_Family", "Entite_N1", "Sexe", "Hot_job", "Anciennete_Bucket"]

feature_cols = numeric_features + categorical_features
X_raw = df[feature_cols].copy()
for c in categorical_features:
    X_raw[c] = X_raw[c].astype(str)

preprocessor = ColumnTransformer([
    ("num", StandardScaler(), numeric_features),
    ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features),
])
X = preprocessor.fit_transform(X_raw)
feature_names = preprocessor.get_feature_names_out()
print("Matrice de features :", X.shape)

**Interprétation.** 48 colonnes en sortie : 14 numériques standardisées + 34 colonnes one-hot (7 `Grade` + 13 `Job_Family` + 5 `Entite_N1` + 2 `Sexe` + 3 `Hot_job` + 4 `Anciennete_Bucket`). C'est cette matrice, et elle seule, qui est fournie aux 3 détecteurs ci-dessous — aucun ne voit `Matricule`, `Job_Title`, ni bien sûr les colonnes de scoring de règles qu'on a supprimées en section 2.

In [ ]:
CONTAMINATION = 0.05

iso = IsolationForest(n_estimators=300, contamination=CONTAMINATION, random_state=RANDOM_STATE)
iso_pred = iso.fit_predict(X)          # -1 = anomalie, 1 = normal
iso_score = -iso.score_samples(X)      # plus haut = plus anormal

lof = LocalOutlierFactor(n_neighbors=35, contamination=CONTAMINATION)
lof_pred = lof.fit_predict(X)
lof_score = -lof.negative_outlier_factor_

ocsvm = OneClassSVM(nu=CONTAMINATION, kernel="rbf", gamma="scale")
ocsvm_pred = ocsvm.fit_predict(X)
ocsvm_score = -ocsvm.decision_function(X)

df["IsoForest_Anomalie"] = (iso_pred == -1).astype(int)
df["LOF_Anomalie"] = (lof_pred == -1).astype(int)
df["OCSVM_Anomalie"] = (ocsvm_pred == -1).astype(int)
df["IsoForest_Score"] = iso_score

for name, col in [("Isolation Forest", "IsoForest_Anomalie"), ("Local Outlier Factor", "LOF_Anomalie"),
                   ("One-Class SVM", "OCSVM_Anomalie")]:
    print(f"{name:22s} : {df[col].sum()} anomalies detectees sur {len(df)} ({df[col].mean():.1%})")

**Interprétation.** Isolation Forest et LOF trouvent tous deux exactement **500 anomalies (5.0%)** : c'est attendu, ces deux modèles utilisent `contamination=0.05` comme un **quantile exact** de leur score — ils sont donc mécaniquement calés sur 5%. One-Class SVM en trouve **507 (5.1%)** : son paramètre `nu` est une *borne* sur la fraction de points hors-frontière, pas un quantile forcé, d'où le léger écart. **Point important à retenir** : ce n'est pas parce que les 3 modèles trouvent ~5% chacun qu'ils sont d'accord sur les **mêmes personnes** — c'est justement ce qu'on va vérifier dans la section suivante.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4.5))
counts = df[["IsoForest_Anomalie", "LOF_Anomalie", "OCSVM_Anomalie"]].sum()
counts.index = ["Isolation Forest", "Local Outlier Factor", "One-Class SVM"]
sns.barplot(x=counts.index, y=counts.values, hue=counts.index, palette="rocket", legend=False, ax=ax)
ax.set_title("Nombre d'anomalies detectees par modele")
ax.set_ylabel("Nombre d'employes flagues")
plt.tight_layout()
plt.show()

**Interprétation.** Visuellement les 3 barres sont quasi identiques (~500-507) — ce graphe confirme juste en un coup d'oeil ce qui vient d'être imprimé. Il ne dit rien sur l'**accord** entre modèles : deux modèles peuvent flaguer 500 personnes chacun sans flaguer les 500 mêmes. C'est l'objet des deux graphes suivants (indice de Jaccard, puis projection PCA).